do DBscan clustering on 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

In [2]:
datasrc = pd.read_csv('../data/students_total_vle.csv')

In [3]:
datasrc.drop(columns=['id_student', 'code_module_x'], inplace=True)

In [4]:
data_clean = datasrc.fillna(datasrc.mean(numeric_only=True))

In [5]:
education_order = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

data_clean['highest_education'] = datasrc['highest_education'].map(education_order)

In [6]:
imd_order = {
    '10-20%': 1, '20-30%': 2, '30-40%': 3, '40-50%': 4, '50-60%': 5,
    '60-70%': 6, '70-80%': 7, '80-90%': 8, '90-100%': 9
}
data_clean['imd_band'] = datasrc['imd_band'].map(imd_order)

In [7]:
final_result_mapping = {
    'Distinction': 3,
    'Pass': 2,
    'Fail': 1,
    'Withdrawn': 0
    }

data_clean['final_result'] = datasrc['final_result'].map(final_result_mapping)

In [8]:
age_band_mapping = {
    '0-35': 1,
    '35-55': 2,
    '55<': 3
}
data_clean['age_band'] = datasrc['age_band'].map(age_band_mapping)

In [9]:
numerical_columns = data_clean.select_dtypes(include=[np.number]).columns

In [10]:
le = LabelEncoder()
data_clean['gender'] = le.fit_transform(data_clean['gender'])
data_clean['disability'] = le.fit_transform(data_clean['disability'])

In [11]:
catigorical_columns = data_clean.select_dtypes(include=['object']).columns
for col in catigorical_columns:
    oh = OneHotEncoder(sparse_output=False, drop='first')
    data_clean_oh = oh.fit_transform(data_clean[[col]])
    cols = [f"{col}_{category}" for category in oh.categories_[0][1:]]
    df_oh = pd.DataFrame(data_clean_oh, columns=cols)
    data_clean = pd.concat([data_clean, df_oh], axis=1)
    data_clean.drop(columns=[col], inplace=True)

In [12]:
data_clean[numerical_columns] = (data_clean[numerical_columns] - data_clean[numerical_columns].mean()) / data_clean[numerical_columns].std()

In [13]:
data_clean.head()

,gender,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avrage_assessment,number_of_assessments_taken,...,region_South West Region,region_Wales,region_West Midlands Region,region_Yorkshire Region,code_module_y_BBB,code_module_y_CCC,code_module_y_DDD,code_module_y_EEE,code_module_y_FFF,code_module_y_GGG
0,1,1.681794,1.647226,NaN,-0.340224,3.901483,0,0.747575,0.632369,-0.629201,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,1.681794,-1.415447,1.559563,-0.340224,-0.481076,0,0.747575,-0.474204,-0.629201,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0.347746,-0.977922,1.559563,-0.340224,-0.481076,1,-1.254538,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0.347746,-0.102873,1.559563,-0.340224,-0.481076,0,0.747575,0.206764,-0.629201,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,-0.986303,-0.102873,-0.641185,-0.340224,-0.481076,0,0.747575,-1.325413,-0.629201,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
data_clean.to_csv('../data/students_cleaned.csv', index=False)